# ChakraNet — Milestone 1: Stage 1 Mesh GNN Anomaly Tracker
**SIH 2026 · PS 26078 · NCMRWF / MoES**

This notebook demonstrates:
1. **Extreme Forecast Index (EFI) & Shift of Tails (SOT)** per M9 mesh node using Lalaurette (2003) integral formulation.
2. **PyTorch Mesh GNN** message passing across spherical geodesic mesh neighbors and temporal GRU tracking.
3. **Dynamic 4D Crop Bounding Box** extraction and cross-member uncertainty cone calculation.
4. **Visual Overlap Comparison** against official IMD/JTWC best-track trajectory for Cyclone Phailin.

In [ ]:
# Colab setup
import sys
from pathlib import Path
root_dir = Path.cwd().resolve()
if root_dir.name == 'notebooks':
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from data.loader import PhailinDataLoader
from data.mesh import IcosahedralMesh
from models.tracker.efi import ExtremeForecastIndex
from models.tracker.gnn_tracker import MeshGNNTracker
from models.tracker.cluster import AnomalyClusterExtractor
from models.tracker.evaluate import plot_milestone1_tracker_cone

### Step 1: Compute Lalaurette (2003) EFI & SOT on M9 Mesh

In [ ]:
loader = PhailinDataLoader()
dataset = loader.load_or_generate_ensemble()
mesh = IcosahedralMesh()

efi_calc = ExtremeForecastIndex(n_quadrature_points=35)
sample_efi = efi_calc.compute_efi(dataset['tp'][:, 9])  # Landfall time
print(f"EFI range at landfall: [{np.min(sample_efi):.3f}, {np.max(sample_efi):.3f}]")
print(f"Anomalous nodes with EFI > 0.65: {np.sum(sample_efi > 0.65)}")

### Step 2: Run Mesh GNN Tracker & Extract 4D Crop Box

In [ ]:
from models.tracker.evaluate import run_stage1_tracker_pipeline
results = run_stage1_tracker_pipeline()

bbox = results['bbox_4d']['bbox_4d']
print("Stage 1 Extracted 4D Bounding Box for Downscaler:")
for k, v in bbox.items():
    print(f"  {k}: {v}")

### Step 3: Render Milestone 1 Visual Overlap Plot

In [ ]:
saved_fig = plot_milestone1_tracker_cone(results)
display(Image(filename=str(saved_fig)))